# 第03讲中文代码说明

按原单元编号阅读。配套讲解见本讲的解释.md，运行方法与记录见复现目录。


## 单元 0：安装命令

三条`!pip install`由IPython转交shell；不是标准Python语法。`-q`减少日志；第一条安装bitsandbytes、datasets、accelerate、loralib，后两条从Git安装Transformers的main与PEFT默认分支。main不固定SHA，原watermark的dev版本无法精确重建。

安装会改环境并联网，不应把本次注释静态核对算作实际安装。原单元没有安装watermark。loralib虽被安装，后面走的是PEFT接口。中文注释说明三条命令各自用途，原命令不改；验证时转换为IPython AST，但不执行这些安装命令。


In [ ]:
!pip install -q bitsandbytes datasets accelerate loralib
!pip install -q git+https://github.com/huggingface/transformers.git@main 
!pip install -q git+https://github.com/huggingface/peft.git

## 单元 1：summary 标题

这个 Markdown 二级标题引出接下来的总体流程。


## summary

## 单元 2：任务与流程概要

原任务是把名言映射到标签文本，模型是BLOOM。LoRA冻结原权重、加低秩参数；PEFT为实现库。MLM通常预测被遮盖位置，CLM预测下一token，本实现选择后者。

原文字把MLM/CLM称作unsupervised，是强调可从文本自动构造token目标，不表示这个有quote/tags关系的任务没有监督信息。pipeline在这里是流程概念，原文件未调用`transformers.pipeline()`。


- bigscience/bloom-7b1
- lora fine-tune bloom: 可插拔式的（plugin/adapter）
    - freeeze original weights
    - plugin lora adapters (peft)
- huggingface transformers 库
    - trainer.train 的参数及过程；
    - mlm 与 clm 的差异：（都是 unsupervised learning，都可以自动地构建 input/labels）
        - mlm：bert
        - clm：gpt（bloom）
    - pipeline
        - dataset/tasks
        - tokenizer
        - training (fine-tune base lora)
        - inference

## 单元 3：base model 与 adapters 标题

Markdown标题区分共享大模型与任务增量；不是Python代码，不执行冻结或注入。


## base model & lora adapters

## 单元 4：导入依赖

torch提供Tensor/autograd，`torch.nn as nn`绑定神经网络模块别名；bitsandbytes绑定bnb，但后续没有直接调用bnb函数，量化由Transformers加载路径触发。三个Auto类按配置选对应的tokenizer/config/因果LM；LoraConfig与get_peft_model分别描述adapter配置和执行注入。

导入依赖包，供后面的模型加载与量化调用使用；bitsandbytes 需要与 CUDA 和相关包版本匹配，环境信息见复现说明。


In [1]:
import torch
import torch.nn as nn
import bitsandbytes as bnb 
from transformers import AutoTokenizer, AutoConfig, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model 


===================================BUG REPORT===================================
Welcome to bitsandbytes. For bug reports, please run

python -m bitsandbytes

 and submit this information together with your error trace to: https://github.com/TimDettmers/bitsandbytes/issues
bin /home/whaow/anaconda3/lib/python3.10/site-packages/bitsandbytes/libbitsandbytes_cuda117.so
CUDA SETUP: CUDA runtime path found: /home/whaow/anaconda3/lib/libcudart.so.11.0
CUDA SETUP: Highest compute capability among GPUs detected: 8.9
CUDA SETUP: Detected CUDA version 117
CUDA SETUP: Loading binary /home/whaow/anaconda3/lib/python3.10/site-packages/bitsandbytes/libbitsandbytes_cuda117.so...


## 单元 5：加载 watermark 扩展

`%load_ext watermark`调用IPython扩展加载机制，需要另有watermark包。状态是当前kernel加载扩展；不改变模型。标准Python脚本直接解析百分号会失败，所以注释核对先做IPython语法转换。


In [3]:
%load_ext watermark

## 单元 6：查看已导入库版本

`%watermark --iversions`打印当前会话导入模块的版本，用于环境记录。没有Tensor输入/输出。保存的历史输出来自老师环境，不是这次CPU环境；没有Git SHA的dev版本字符串不能固定源码。


In [4]:
%watermark --iversions

torch       : 2.0.0+cu118
bitsandbytes: 0.38.1



## 单元 7：函数方式查看环境

从 watermark 导入同名函数，packages 接收逗号分隔的包名字符串，返回版本报告供 print 输出。它与前格 magic 提供两种调用方式；量化组件的版本可另行加入 packages 查询。


In [9]:
from watermark import watermark
print(watermark(packages='peft,torch,loralib,transformers,accelerate,datasets'))

peft        : 0.4.0.dev0
torch       : 2.0.0+cu118
loralib     : 0.1.1
transformers: 4.30.0.dev0
accelerate  : 0.18.0
datasets    : 2.11.0



## 单元 8：加载模型与 tokenizer

AutoModelForCausalLM.from_pretrained为类方法，输入模型ID字符串、load_in_8bit=True、device_map='auto'，返回因果LM模块；可能下载权重、分配设备内存和设置dispatch hook。AutoTokenizer.from_pretrained单独加载匹配的分词器，不加载Transformer权重。

8bit是旧式bitsandbytes量化入口，不是所有参数/激活均为int8；auto是设备分派策略，不是数据并行副本数。这里未指定revision，无法精确锁定模型/词表快照。三个加载参数与tokenizer调用的作用见相邻注释，运行条件见复现说明。


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    "bigscience/bloom-7b1", 
    load_in_8bit=True, 
    device_map='auto',
)

tokenizer = AutoTokenizer.from_pretrained("bigscience/bloom-7b1")

## 单元 9：查看配置

第一行`# model.config`未执行；实际调用AutoConfig.from_pretrained返回配置对象。它读取配置并返回独立配置对象，供后面的属性查看使用。参数值应以固定模型revision核对，原文件缺revision。


In [ ]:
# model.config
AutoConfig.from_pretrained("bigscience/bloom-7b1")

## 单元 10：查看原模型结构

末尾表达式 model 让 notebook 展示模型树；普通脚本中可使用 print(model)。沿模型树找到融合的 query_key_value 投影。


In [ ]:
model

## 单元 11：查看输入 embedding

注释里的具体属性访问不执行；实际`model.get_input_embeddings()`是实例方法，返回输入Embedding模块，未对任何ID做查表。其权重通常为`[V,H]`，实际lookup将int64输入`[B,S]`变成浮点激活`[B,S,H]`。

本格不解冻权重；返回模块也不是复制一份参数。默认参数为空，模块无输入Tensor。


In [ ]:
# model.transformer.word_embeddings
model.get_input_embeddings()

## 单元 12：查看 tokenizer

展示 tokenizer 的配置、特殊 token 等信息；下一步传入文本时再进行编码。


In [ ]:
tokenizer

## 单元 13：冻结标题

Markdown标题指出接下来冻结基权重。标题本身不改变requires_grad。


### freeze original weights

## 单元 14：只看首个参数 dtype

model.parameters返回迭代器，list将其收集为参数引用列表，`[0]`取第一个Parameter，`.dtype`返回类型对象。参数必须存在。

只观察第一项，不能把结果推广到全部Linear8bitLt/norm/adapter。没有Tensor拷贝或dtype转换，后格才实际修改。


In [ ]:
list(model.parameters())[0].dtype

## 单元 15：冻结和一维参数转 FP32

`enumerate(model.parameters())`给每个Parameter提供编号i和param；i只用于原注释中的调试打印，没有参与计算。循环将requires_grad=False，再用ndim==1选择一维权重/偏置，把底层data转为float32。shape保持，参数存储/类型改变。

一维条件不是“所有小参数”或“只有LayerNorm”。二维Embedding仍被冻结而不被此条件cast；冻结不等于跳过其计算。`.data`绕过一般autograd跟踪，是原历史写法；注释版保留，未擅自改算法。本次在小型真实BLOOM上执行该循环，核对所有基参数冻结。


In [ ]:
for i, param in enumerate(model.parameters()):
    param.requires_grad = False  # freeze the model - train adapters later
#     print(i, 'param.requires_grad = False')
    if param.ndim == 1:
        # cast the small parameters (e.g. layernorm) to fp32 for stability
        param.data = param.data.to(torch.float32)
#         print(i, 'ndim == 1, torch.float16 to torch.float32')

## 单元 16：checkpointing 与输入激活梯度

两个实例方法无显式参数。gradient_checkpointing_enable改变模型的激活保存/重算设置；不把训练checkpoint写入磁盘。enable_input_require_grads安装hook，使Embedding输出激活requires_grad=True；不会把Embedding.weight解冻。

目的在于冻结模型与激活重算时维持adapter所需梯度链。实际CPU验证核对Embedding权重仍False，而输出激活为True，并检查小模型开启重算标志。


In [ ]:
# reduce number of stored activations
model.gradient_checkpointing_enable()  
model.enable_input_require_grads()

## 单元 17：lm_head 输出转 FP32

CastOutputToFloat是nn.Sequential子类；forward(self,x)覆盖父方法，super().forward先运行被包装lm_head，再把返回Tensor转float32。它接收`[B,S,H]`，返回`[B,S,V]`，shape不因cast改变。

把model.lm_head赋为包装对象修改模块树，原head变成其子模块；不是创建另一个词表，也不是解冻head。class/继承/self/super分别是类型定义、复用父实现、当前实例和父方法访问。没有自写__init__，沿用Sequential构造行为。


In [ ]:
class CastOutputToFloat(nn.Sequential):
    def forward(self, x): 
        return super().forward(x).to(torch.float32)
model.lm_head = CastOutputToFloat(model.lm_head)

## 单元 18：LoRA 标题

Markdown过渡到adapter部分，本格未注入任何矩阵。


### LoRa Adapters

## 单元 19：统计可训练参数的函数

print_trainable_parameters(model)是独立函数，无默认参数。named_parameters迭代名字与Parameter，`_`忽略名字；numel统计元素数。all_param计全部，trainable_params仅计requires_grad=True，再用f-string打印比例。返回None，不改参数。

第一行docstring是函数说明str，不是运行命令；定义时不会调用统计循环，单元21才调用。全空模型会除零，原函数未处理；真实BLOOM有参数。量化参数元素计数不是实际显存字节数。


In [ ]:
def print_trainable_parameters(model):
    """
    Prints the number of trainable parameters in the model.
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}"
    )

## 单元 20：LoRA 配置

重复导入一般复用模块缓存，不是重复安装。LoraConfig构造配置对象：r=16，lora_alpha=32，标准scale=32/16=2；dropout=0.05只用于adapter支路；bias='none'不额外训练偏置；task_type='CAUSAL_LM'选择因果LM包装。

target_modules 这一行处于注释中，当前 BLOOM 使用默认的 query_key_value 目标映射。r 决定低秩矩阵的尺寸，alpha 决定修正分支的缩放。


In [ ]:
from peft import LoraConfig, get_peft_model 
config = LoraConfig(
    r=16, #low rank
    lora_alpha=32, #alpha scaling， scale lora weights/outputs
    # target_modules=["q_proj", "v_proj"], #if you know the 
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM" # set this for CLM or Seq2Seq
)

## 单元 21：注入并统计参数

get_peft_model(model,config)返回PEFT包装，并改造原模型内部目标模块；赋回model名字，因此后续调用使用adapter模型。不是对原模型深拷贝。随后调用前格函数打印计数。

原7B的融合QKV权重方向为`[12288,4096]`，新增A`[16,4096]`、B`[12288,16]`；激活为`[B,S,4096]→[B,S,16]→[B,S,12288]`再与基输出相加。每层新增262144个参数，30层推得7864320；真实目标/计数仍应检查版本与模型。新CPU验证使用2层H=16的随机模型，结果按它的真实尺寸单独记录。


In [ ]:
model = get_peft_model(model, config)
print_trainable_parameters(model)

## 单元 22：查看包装后的模型

展示注入 LoRA 后的 model，沿 PeftModel 包装找到新增模块。可结合参数遍历和 requires_grad 查看哪些权重参与训练。


In [ ]:
model

## 单元 23：pipeline 标题

Markdown流程标题，不调用任何pipeline类或函数。


## pipeline

## 单元 24：data 标题

Markdown引出数据检查，尚未下载dataset。


### data

## 单元 25：加载名言数据

import transformers绑定包名，后面trainer接口使用；from datasets导入load_dataset。load_dataset('Abirate/english_quotes')可能联网、缓存并返回DatasetDict；原调用没固定revision，也没指定split，后面取train。

数据字段quote/author通常为str，tags为list[str]。读取后得到Dataset中的Python对象；配套算例使用5条自编记录展示取行、取列等Dataset操作。


In [ ]:
import transformers
from datasets import load_dataset
dataset = load_dataset("Abirate/english_quotes")

## 单元 26：查看 DatasetDict

dataset 展示各个 split、字段和行数。DatasetDict 将 split 名称映射到相应 Dataset，随后再进行文本处理和分词。


In [ ]:
dataset

## 单元 27：取得 train split

dataset['train']按名称取Dataset；与list索引不同。必须存在train键，缺少会报KeyError。返回数据集对象，不是某一行，也未转Tensor。


In [ ]:
dataset['train']

## 单元 28：转 pandas 表格

to_pandas返回DataFrame，便于查看列和行；可能物化大量数据到内存。不会改变Dataset字段，也不会把数据送到GPU。大数据不应默认全部转表。


In [ ]:
dataset['train'].to_pandas()

## 单元 29：quote 列的前四条

先`['quote']`取列，再`[:4]`切前四项，结果为list[str]。从0开始、不含右端点4。列表的每一项是一条完整名言文本。


In [ ]:
dataset['train']['quote'][:4]

## 单元 30：author 列的前四条

与前格相同的取列再切片，返回作者字符串列表。后面的merge选用quote和tags构造训练文本。


In [ ]:
dataset['train']['author'][:4]

## 单元 31：同时看前四条的所有列

Dataset的`[:4]`按行批取，结果为dict，各列值是长度4的列表；不是list[dict]。与单元29只取quote列不同。后续batched map同样处理按列组织的批次字典。


In [ ]:
dataset['train'][:4]

## 单元 32：把标签列表变成字符串

先取tags列再取第0条list[str]，str将其转成带方括号/引号的Python列表展示文本。不是JSON序列化、不是类别ID，也未tokenize。空列表会成为'[]'，之后模型学的是这类字符串形式。


In [ ]:
str(dataset['train']['tags'][0])

## 单元 33：merge 函数与 Dataset.map

merge(row) 接收单行 dict，取 quote，加上固定分隔符 ' ->: ' 和 str(tags)，把拼接结果写入 prediction 字段，再返回该记录。这里 prediction 是整理好的训练文本。

Dataset.map逐行调用函数并返回新Dataset，再赋回train split；原API不是对整个Dataset直接原地添列。其他split若存在不会在此合并，后续DatasetDict.map访问prediction时可能失败。作者字段保留但没有拼进文本。

本次提取原函数实际执行，例如quote='Be yourself.'、tags=['life']得到`Be yourself. ->: ['life']`，并核对传入dict被修改。后续对新编数据运行真实map。


In [ ]:
def merge(row):
    row['prediction'] = row['quote'] + ' ->: ' + str(row['tags'])
    return row
dataset['train'] = dataset['train'].map(merge)

## 单元 34：查看拼接后前五条

取prediction列再[:5]，结果为list[str]，比前面检查多一条。要求单元33已成功创建字段；没有model.generate。


In [ ]:
dataset['train']['prediction'][:5]

## 单元 35：查看第五条记录

Dataset整数索引[4]返回单行dict，各字段为标量/列表，和[:4]的列列表字典不同。五条以下的数据会越界；本次教学数据明确有5条。


In [ ]:
dataset['train'][4]

## 单元 36：试分词前四条

调用tokenizer对象的__call__，输入list[str]，返回BatchEncoding，默认是input_ids/attention_mask等列表，不是torch Tensor。原调用没padding/truncation，长度可不同，未截断超长序列。

默认可能加入该词表定义的特殊token，精确ID取决于tokenizer快照。试调用没有把返回值赋回dataset；正式转换在38。新教学tokenizer不是BLOOM真实词表，ID仅用于API和loss路径验证。


In [ ]:
tokenizer(dataset['train']['prediction'][:4])

## 单元 37：tokenize 标题

Markdown标题标记正式数据转换阶段，无函数调用。


### tokenize

## 单元 38：批量 map 分词

lambda samples为匿名函数，取samples['prediction']字符串列表，调用tokenizer。batched=True让map传入列列表字典，默认非此文件指定的固定微批大小；不等于训练batch_size=4。

dataset为DatasetDict，map对所有split执行并返回新对象；添加input_ids与attention_mask列，原quote/tags等列默认仍保留。因未指定padding/truncation，后面collator负责动态补齐。Trainer通常移除forward不接收的文本列；直接调用collator时应只传模型输入字段。


In [ ]:
dataset = dataset.map(lambda samples: tokenizer(samples['prediction']), batched=True)

## 单元 39：查看分词后数据集

展示分词后的数据结构，重点看 input_ids 与 attention_mask 两列；数据整理器随后将列表组合成训练张量。


In [ ]:
# 'input_ids', 'attention_mask'
dataset

## 单元 40：training 标题

Markdown转入训练，标题不更新权重。


### training

## 单元 41：导入训练接口

Trainer组织训练，TrainingArguments保存调度/批次/精度等配置，DataCollatorForLanguageModeling构造语言模型批次和labels。导入不开始训练，也不是定义模型loss公式；BLOOM内部处理CLM位移。


In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

## 单元 42：构造 Trainer 并训练

Trainer接收model、train Dataset、TrainingArguments和collator。原参数无独立eval_dataset，不能从训练日志推出验证集效果。

| 参数 | 值与含义 | 容易混淆的边界 |
|---|---|---|
| per_device_train_batch_size | 4，每设备微批样本数 | 不是token长度 |
| gradient_accumulation_steps | 4，累积四个微批再更新 | 单个完整周期单设备有效batch为16 |
| warmup_steps | 100，优化step预热数 | 不是前100条样本 |
| max_steps | 200，优化更新上限 | 不是固定200个微批 |
| learning_rate | 2e-4 | 与LoRA scale=2不同 |
| fp16 | True，训练混合精度 | 不表示每个参数都为fp16 |
| logging_steps | 1，更新日志间隔 | 不保证额外评估 |
| output_dir | outputs，训练输出目录 | 无显式save_pretrained导出流程 |

mlm=False选择CLM collator：动态padding后input_ids/attention_mask为int64`[B,Smax]`，labels复制ID并将padding目标设-100。原BLOOM内部logits`[B,S,V]`向左切一格，与labels右移一格配对计算交叉熵；不会让当前位置预测自己。全段非padding文本计loss，不仅标签回答段。

model.config.use_cache=False修改配置，使训练与checkpointing配合；不冻结参数或关闭autograd。trainer.train实际组织forward/loss/backward/累积与optimizer更新，改变adapter和训练状态并可能写文件。auto设备分派不是数据并行world_size，不能直接按GPU数扩大有效batch。

配套 CPU 算例使用随机小型 BLOOM 和真实 Trainer 训练2步：fp16=False、微批2、累积2、warmup0。原训练配置及历史输出保留在 notebook 中，运行记录见复现说明。


In [ ]:
trainer = Trainer(
    model=model, 
    train_dataset=dataset['train'],
    args=TrainingArguments(
        per_device_train_batch_size=4, 
        gradient_accumulation_steps=4,
        warmup_steps=100, 
        max_steps=200, 
        learning_rate=2e-4, 
        fp16=True,
        logging_steps=1, 
        output_dir='outputs'
    ),
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)
model.config.use_cache = False  
trainer.train()

## 单元 43：inference 标题

Markdown开始生成示例，未自动切换eval模式或关闭梯度。


### inference

## 单元 44：第一条生成示例

tokenizer 接收带有 ' ->: ' 的字符串，return_tensors='pt' 使 BatchEncoding 内的编号成为 [1,S] 整数张量。CUDA autocast 控制浮点运算的精度；输入设备由后续迁移操作决定。

`**batch`把input_ids/attention_mask作为关键字实参传给model.generate；max_new_tokens=50限制新增token，输出整数`[1,S+G]`且G≤50。`output_tokens[0]`取一行；decode(...,skip_special_tokens=True)返回str，通常仍包含输入前缀。print额外输出换行。

生成前检查 model 的模式、输入设备与 dispatch hook。CPU 配套例子显式调用 eval 和 no_grad，生成3个新增 token，便于观察输入前缀与续写结果。


In [ ]:
batch = tokenizer("“Training models with PEFT and LoRa is cool” ->: ", return_tensors='pt')

with torch.cuda.amp.autocast():
    output_tokens = model.generate(**batch, max_new_tokens=50)

print('\n\n', tokenizer.decode(output_tokens[0], skip_special_tokens=True))

## 单元 45：第二条生成示例

与44完全相同的API路径，只把prompt换成更长的论文风格句子。开头空白行保留；长文本使S变化，不是新的模型、独立验证集或分类head。

字符串中的引号、标点也被tokenize。max_new_tokens仍50，autocast仍不搬设备，decode仍包含前缀。两条prompt都需解释，不能用“同上”替代本格的输入差别及评估边界。


In [ ]:
 
batch = tokenizer("“An important paradigm of natural language processing consists of large-scale pre-training on general domain data and adaptation to particular tasks or domains.” ->: ", return_tensors='pt')

with torch.cuda.amp.autocast():
    output_tokens = model.generate(**batch, max_new_tokens=50)

print('\n\n', tokenizer.decode(output_tokens[0], skip_special_tokens=True))

## 单元 46：查看 collator

访问trainer.data_collator并显示其类型和配置。调用这个对象并传入样本后，它才会把样本整理成训练批次。

实际batch转换是调用该collator(samples)，需要样本列表；新验证通过真实collator检查padding labels=-100、shift与BLOOM返回loss一致。


In [ ]:
trainer.data_collator